In [47]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score , precision_score, recall_score
import torch
from transformers import TrainerCallback
from torch.utils.data import DataLoader, Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)
import warnings
warnings.filterwarnings('ignore')

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [18]:
df = pd.read_csv('cellula toxic data  (1).csv')

In [19]:
del df['image descriptions']

# TEXT PREPROCESSING

In [20]:
def preprocess_text_bert(text):
    """
    Basic preprocessing for BERT (preserves BERT's tokenization patterns)
    """

    text = str(text).lower()

    text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
    return text

In [21]:
df['query_clean'] = df['query'].apply(preprocess_text_bert)

# LABEL ENCODING

In [22]:
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['Toxic Category'])
num_classes = len(label_encoder.classes_)

# DATA SPLITTING

In [23]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label_encoded']
)

# TOKENIZER INITIALIZATION

In [24]:
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
max_length = 128

# CUSTOM DATASET CLASS

In [55]:
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# DATASET CREATION

In [56]:
train_dataset = ToxicDataset(train_df['query_clean'], train_df['label_encoded'], tokenizer, max_length)
test_dataset = ToxicDataset(test_df['query_clean'], test_df['label_encoded'], tokenizer, max_length)

# MODEL LOADING

In [57]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    id2label={i: label for i, label in enumerate(label_encoder.classes_)},
    label2id={label: i for i, label in enumerate(label_encoder.classes_)}
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# LoRA CONFIGURATION

In [58]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    # bias="none",  # Don't train bias parameters
)

# APPLYING LoRA

In [59]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 744,969 || all params: 67,705,362 || trainable%: 1.1003


# METRICS FUNCTION

In [60]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_micro = f1_score(labels, predictions, average='micro')
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    recall = recall_score(labels, predictions, average='weighted')
    precision= precision_score(labels, predictions, average='weighted')

    return {
        "accuracy": accuracy,
        "recall": recall ,
        "precision": precision,
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

# TRAINING ARGUMENTS

In [61]:
training_args = TrainingArguments(
    output_dir='./distilbert-lora-toxic',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=2,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# TRAINER SETUP

In [62]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [63]:
train_history = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Recall,Precision,F1 Micro,F1 Macro,F1 Weighted
1,4.203842,1.821986,0.403333,0.403333,0.236063,0.403333,0.101617,0.280228
2,3.088638,1.402323,0.486667,0.486667,0.372599,0.486667,0.199188,0.396594
3,2.668008,1.198636,0.646667,0.646667,0.587849,0.646667,0.615589,0.598354
4,2.331739,1.088736,0.720000,0.720000,0.761210,0.720000,0.792138,0.709023
5,2.183123,1.051473,0.720000,0.720000,0.759941,0.720000,0.791656,0.709216


# EVALUATION

In [68]:
test_results = trainer.evaluate(eval_dataset=test_dataset)
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['label_encoded'].values

In [71]:
# Calculate metrics
test_accuracy = accuracy_score(true_labels, pred_labels)
test_f1_micro = f1_score(true_labels, pred_labels, average='micro')
test_recall=recall_score(true_labels,pred_labels, average='weighted')
test_precision=precision_score(true_labels,pred_labels, average='weighted')

print(f"\n📊 TEST SET METRICS:")
print("-" * 40)
print(f"Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f} %)")
print(f"F1 Micro:  {test_f1_micro:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"Precision: {test_precision:.4f}")
print("-" * 40)



📊 TEST SET METRICS:
----------------------------------------
Accuracy:  0.7200 (72.00 %)
F1 Micro:  0.7200
Recall:    0.7200
Precision: 0.7612
----------------------------------------
